In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import geometry_mask
from rasterio.mask import mask

from paths import paths, prepare_preprocessing
from preprocessing.src.constants import CRS_EUROPEAN, LSOA_IDS, REGION
from preprocessing.src.geo import get_buffered_region
from preprocessing.src.isochrone import extract_zones_by_duration
from preprocessing.src.threading import parallelize

In [ ]:
prepare_preprocessing()

In [ ]:
TIFF_PATH = paths.raw.walkability.land_use

CODE_TO_GENERAL_USE = {
    0: 'no_data',
    5: 'blue_space',
    11: 'residential',
    12: 'industrial',
    13: 'disturbance',
    14: 'green_space',
    21: 'green_space',
    22: 'green_space',
    23: 'green_space',
    31: 'green_space',
    32: 'green_space',
    33: 'green_space',
    41: 'blue_space',
    42: 'blue_space'
}

In [ ]:
area = get_buffered_region(REGION, 5000, CRS_EUROPEAN)

In [ ]:
def add_land_use_score(gdf):
    gdf['land_use_score'] = (gdf['entropy'] - gdf['entropy'].mean()) / gdf['entropy'].std(ddof = 0)

    return gdf

def process_lsoa(lsoa):
    zones_by_duration = extract_zones_by_duration(lsoa, buffer = 15, crs = CRS_EUROPEAN)
    windows = {}
    results = {}

    with rasterio.open(TIFF_PATH) as tif:
        nodata = tif.nodata or 0

        for minutes, zones in zones_by_duration.items():
            entropy_list = []
            landuse_mix_list = []

            for seed, zone in zones:
                # Read the 20-minute window once, then narrow its mask for 15 and 10 minutes.
                if seed not in windows:
                    data, transform = mask(tif, [zone], crop = True)
                    band = data[0]
                    valid = (band != nodata) & np.isin(band, list(CODE_TO_GENERAL_USE.keys()))
                else:
                    band, transform, valid = windows[seed]
                    valid = valid & geometry_mask([zone], out_shape = band.shape, transform = transform, invert = True)

                windows[seed] = (band, transform, valid)
                vals = band[valid]
                
                unique, counts = np.unique(vals, return_counts = True)
                total = counts.sum()
                proportions = counts / total
                entropy_list.append(-(proportions * np.log(proportions)).sum())
                mix = {use: 0.0 for use in CODE_TO_GENERAL_USE.values()}

                for code, count in zip(unique, counts):
                    mix[CODE_TO_GENERAL_USE[code]] += count / total

                landuse_mix_list.append(mix)

            # Give each seed neighbourhood equal weight in the LSOA averages
            results[minutes] = (lsoa, float(np.mean(entropy_list)), pd.DataFrame(landuse_mix_list).mean().to_dict())

    return results

In [ ]:
results = parallelize(process_lsoa, LSOA_IDS)

In [ ]:
lsoa_base = gpd.read_parquet(paths.preprocessing.shared.lsoa)
results_by_duration = {minutes: [result[minutes] for result in results] for minutes in (20, 15, 10)}

In [ ]:
component_data = {}

for minutes, duration_results in results_by_duration.items():
    entropy_per_lsoa = {result[0]: result[1] for result in duration_results}
    landuse_mix_per_lsoa = {result[0]: result[2] for result in duration_results}
    
    # Load LSOA data and add entropy and land use fractions
    lsoa = lsoa_base.copy()
    lsoa['entropy'] = lsoa['LSOA21CD'].map(entropy_per_lsoa).fillna(0).astype(float)

    for key in CODE_TO_GENERAL_USE.keys():
        lsoa[f'land_use_fraction_{CODE_TO_GENERAL_USE[key]}'] = (
            lsoa['LSOA21CD']
            .map(lambda x: landuse_mix_per_lsoa[x].get(CODE_TO_GENERAL_USE[key], 0))
            .fillna(0)
            .astype(float)
        )

    lsoa = add_land_use_score(lsoa)
    component_data[minutes] = lsoa

In [ ]:
columns = [
    'LSOA21CD',
    'geometry',
    'land_use_score',
    *[f'land_use_fraction_{use}' for use in set(CODE_TO_GENERAL_USE.values())]
]

for minutes, lsoa in component_data.items():
    lsoa[columns].to_parquet(paths.preprocessing.walkability[minutes].land_use, index = False)

In [ ]:
for minutes, lsoa in component_data.items():
    FILES = paths.preprocessing.walkability[minutes]
    
    m = lsoa.explore(
        column = 'land_use_score',
        cmap = 'viridis',
        scheme = 'quantiles',
        k = 50,
        legend = True,
        tiles = 'CartoDB positron',
        style_kwds = {'weight': 0.5, 'lineOpacity': 0.5, 'fillOpacity': 0.4},
    )

    m.save(FILES.land_use_map)

m